# Shape: profile, check and diff a lakehouse table (PySpark notebook)

Attach the **Shape Environment** (Runtime 2.0, custom library `sqllocks_shape` wheel;
see `integrations/fabric/environment/README.md`) and the default lakehouse. Nothing is
installed in the notebook itself.

The table is read with `spark.read.table(...)` and profiled on the **driver**
(Shape's exact mode). Driver-side profiling is bounded by `DRIVER_ROW_LIMIT` rows (see
the next cells): above it, the table is sampled and the result carries `sampled: true`.
The distributed bounded mode arrives in PF-02.

Same parameters, artifacts and exit value as `shape_profile.ipynb`.


In [ ]:
# Parameters cell (toggle "parameter cell" in Fabric). A pipeline overrides these.
# Paths are relative to the lakehouse Files/ folder, or absolute.
tableName = "orders"      # Delta table in the default lakehouse ("schema.table" also accepted)
contractPath = ""         # optional contract JSON, for example "contracts/orders.json"
baselinePath = ""         # optional earlier .shape artifact to diff against
outputDir = "shape"       # artifacts go to Files/<outputDir>/<table>/<timestamp>/
failOnDrift = False       # True: drift against the baseline also fails the gate


In [ ]:
# Driver-side profiling limit. The Arrow copy of the table lives in driver memory
# (roughly 100-200 bytes per cell for mixed types), so the default is 5,000,000 rows,
# which is comfortable on a Medium (8 vCore / 64 GB) node. Lower it for wide tables.
DRIVER_ROW_LIMIT = 5_000_000
SAMPLE_SEED = 42
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import inspect

import shape
import pyarrow as pa

LAKEHOUSE = "/lakehouse/default"
FILES = f"{LAKEHOUSE}/Files"


def _as_bool(value) -> bool:
    """Pipeline parameters may arrive as strings."""
    if isinstance(value, str):
        return value.strip().lower() in ("1", "true", "yes", "y")
    return bool(value)


def _resolve(path: str) -> str:
    return path if os.path.isabs(path) else f"{FILES}/{path}"


failOnDrift = _as_bool(failOnDrift)
safe_name = str(tableName).replace(".", "_")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
out_rel = f"{outputDir.strip('/')}/{safe_name}/{stamp}"
out_dir = Path(FILES) / out_rel
MAX_LISTED = 100  # keep the exit value small (well under 1 MB)


In [ ]:
df = spark.read.table(str(tableName))
total_rows = df.count()
sampled = total_rows > DRIVER_ROW_LIMIT
if sampled:
    df = df.sample(withReplacement=False, fraction=DRIVER_ROW_LIMIT / total_rows, seed=SAMPLE_SEED)

# Spark 4 (Runtime 2.0) has DataFrame.toArrow(); Spark 3.5 (Runtime 1.3) goes through pandas.
if int(spark.version.split(".")[0]) >= 4:
    table = df.toArrow()
else:
    table = pa.Table.from_pandas(df.toPandas(), preserve_index=False)

# The section 12.2 API is profile(source, *, name=None). Ask for exact mode only when the
# installed Shape offers it.
kwargs = {"name": str(tableName)}
if "exact" in inspect.signature(shape.profile).parameters:
    kwargs["exact"] = True
profile = shape.profile(table, **kwargs)
print(f"Profiled {table.num_rows:,} of {total_rows:,} rows (sampled={sampled}); Spark {spark.version}")


In [ ]:
violations, changes, drifted = [], [], False

if contractPath:
    check_result = shape.check(profile, _resolve(contractPath))
    violations = list(check_result.violations)

if baselinePath:
    diff_result = shape.diff(shape.load(_resolve(baselinePath)), profile)
    drifted = bool(diff_result.drifted)
    changes = list(diff_result.changes)
    if drifted and failOnDrift:
        violations.append(
            {
                "column": "*",
                "rule": "drift",
                "expected": "no drift against baseline",
                "observed": f"{len(changes)} change(s)",
            }
        )

passed = not violations


In [ ]:
out_dir.mkdir(parents=True, exist_ok=True)
shape_file = out_dir / f"{safe_name}.shape"
shape.save(profile, str(shape_file))
(out_dir / f"{safe_name}.html").write_text(profile.to_html(), encoding="utf-8")
(out_dir / f"{safe_name}.summary.json").write_text(json.dumps(profile.summary()), encoding="utf-8")

artifact_path = f"{out_rel}/{safe_name}.shape"  # relative to Files/, usable as baselinePath
print("Artifacts written to", out_dir)


In [ ]:
from IPython.display import HTML, display

display(HTML(profile.to_html()))


In [ ]:
result = {
    "table": tableName,
    "rows": total_rows,
    "passed": passed,
    "violations": violations[:MAX_LISTED],
    "drifted": drifted,
    "changes": changes[:MAX_LISTED],
    "artifactPath": artifact_path,
    "sampled": sampled,
    "truncated": len(violations) > MAX_LISTED or len(changes) > MAX_LISTED,
}
print(json.dumps(result, indent=2, default=str)[:4000])


In [ ]:
import notebookutils

notebookutils.notebook.exit(json.dumps(result, default=str))
